# Pipeline End-to-End — Multi-Intent Agentic Planning

Notebook chay toan bo pipeline EduBot voi **Multi-Intent Detection**, phan ro **tung node** de kiem soat:

```
1 ContextAnalyzer -> 2 IntentRouter (LLM, multi-intent) -> 3 SessionManager -> 4 ActionPlanner (multi-plan) -> 5 RAG Search -> 6 Handler(s) -> 7 Session Save
```

Ket qua pipeline duoc ghi ra `pipeline_trace.log` dang **JSON** (append mode). File `app.log` giu persistent log.

---
## 0. Setup — Path & Environment

In [1]:
import sys
import os
import time
import json
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent.parent
print(f"Project root: {PROJECT_ROOT}")

# Add src to path
for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Load env
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

print(f"API Key: {'SET' if os.getenv('GENAI_API_KEY') else 'NOT SET'}")
print(f"Python: {sys.version.split()[0]}")

Project root: c:\Users\Admin\OneDrive - Hanoi University of Science and Technology\Desktop\DATN
API Key: SET
Python: 3.12.4


---
## 1. Init — CustomSearch + Reranker + Orchestrator

In [2]:
from src.config.config import settings
from src.rag.retrieve_rebuild import CustomSearch
from src.rag.reranker import Reranker
from src.llm.orchestrator import Orchestrator

DATA_DIR = PROJECT_ROOT / 'data'
CHUNKS_PATH = str(DATA_DIR / 'rag_chunks_v2.json')
EMBEDDINGS_PATH = str(DATA_DIR / 'embeddings.npy')

print("=" * 60)
print("[NODE 0] Initializing Components")
print("=" * 60)

# 1a. CustomSearch
t0 = time.time()
searcher = CustomSearch(chunks_path=CHUNKS_PATH, embeddings_path=EMBEDDINGS_PATH)
print(f"  CustomSearch: {searcher.corpus_size} chunks, dim={searcher.embeddings.shape[1]} ({time.time()-t0:.2f}s)")

# 1b. Reranker
reranker = Reranker()
print(f"  Reranker: {settings.RERANKER_MODEL} (lazy load)")

# 1c. Orchestrator
orch = Orchestrator(retriever=searcher, reranker=reranker)
print(f"  Orchestrator: ready")
print(f"  LLM Model: {settings.LLM_MODEL}")
print("=" * 60)

[15:51:38] INFO    | Tokenizing 2348 docs with underthesea...


[NODE 0] Initializing Components
CustomSearch initialized: 2348 docs, vocab=9672, avgdl=137.7
  CustomSearch: 2348 chunks, dim=768 (15.09s)
  Reranker: AITeamVN/Vietnamese_Reranker (lazy load)
  Orchestrator: ready
  LLM Model: gemini-2.5-flash-lite


---
## 2. Helpers — JSON Viewer & Query Runner

In [3]:
TRACE_LOG = PROJECT_ROOT / 'logs' / 'pipeline_trace.log'
APP_LOG = PROJECT_ROOT / 'logs' / 'app.log'


def pp(obj, title=""):
    """Pretty-print JSON."""
    if title:
        print(f"\n{'\u2500'*60}")
        print(f"  {title}")
        print(f"{'\u2500'*60}")
    print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))


def parse_step(debug_info: dict, node_name: str) -> dict:
    """Extract a specific node from debug_steps."""
    for s in debug_info.get("steps", []):
        if s.get("node") == node_name:
            return s
    return {}


def parse_all_steps(debug_info: dict, node_name: str) -> list:
    """Extract ALL steps matching a node name."""
    return [s for s in debug_info.get("steps", []) if s.get("node") == node_name]


def run_query(orch_inst, query: str, user_id: str = "test_user",
              ui_book: str = "KNTT") -> dict:
    """
    Run 1 query qua full pipeline.
    Tra ve JSON day du voi tat ca node outputs.
    """
    t0 = time.time()
    chunks = []
    for chunk in orch_inst.ask(query, ui_book=ui_book, user_id=user_id):
        chunks.append(chunk)
    elapsed = time.time() - t0

    response = "".join(chunks)
    debug = orch_inst.last_debug_info or {}

    # Per-node extraction
    ctx_step = parse_step(debug, "ContextAnalyzer")
    intent_step = parse_step(debug, "IntentRouter")
    session_step = parse_step(debug, "SessionManager")
    action_step = parse_step(debug, "ActionPlanner")
    handler_steps = parse_all_steps(debug, "Handler")
    book_step = parse_step(debug, "BookFilter")

    return {
        "meta": {
            "request_id": debug.get("request_id"),
            "user_id": debug.get("user_id"),
            "query": query,
            "ui_book": ui_book,
            "timestamp": debug.get("timestamp"),
            "total_time_s": debug.get("total_time_s", round(elapsed, 2)),
        },
        "pipeline": {
            "1_context_analyzer": {
                "enriched": ctx_step.get("enriched", False),
                "rewrite": ctx_step.get("rewrite"),
            },
            "2_intent_router": {
                "total_intents": intent_step.get("total_intents", 1),
                "intents": intent_step.get("intents", [
                    {
                        "intent": intent_step.get("primary_intent"),
                        "task_type": intent_step.get("task_type"),
                        "topic": intent_step.get("topic"),
                        "is_new_topic": intent_step.get("is_new_topic"),
                        "book": intent_step.get("book"),
                    }
                ]),
                "time_s": intent_step.get("time_s"),
            },
            "3_session_manager": {
                "session_id": session_step.get("session_id"),
                "user_id": session_step.get("user_id"),
                "topic": session_step.get("topic"),
                "intent": session_step.get("intent"),
                "book": session_step.get("book"),
                "total_messages": session_step.get("total_messages"),
                "has_quiz_state": session_step.get("has_quiz_state"),
                "has_slide_state": session_step.get("has_slide_state"),
            },
            "4_action_planner": {
                "total_plans": action_step.get("total_plans", 1),
                "plans": action_step.get("plans", [
                    {
                        "action": action_step.get("action"),
                        "reason": action_step.get("reason"),
                        "round_id": action_step.get("round_id"),
                    }
                ]),
            },
            "5_book_resolution": {
                "effective_book": debug.get("effective_book"),
                "book_filter_status": book_step.get("status", "passed"),
            },
            "6_handlers": handler_steps or [{"status": "no_handler_logged"}],
        },
        "response": {
            "length": len(response),
            "has_separator": "---" in response,
            "num_sections": response.count("---") + 1 if "---" in response else 1,
            "preview": response[:500],
        },
        "raw_debug_steps": debug.get("steps", []),
    }


def show_app_log(n=30):
    """Hien thi n dong cuoi cua app.log."""
    if APP_LOG.exists():
        lines = APP_LOG.read_text(encoding='utf-8').strip().split('\n')
        for line in lines[-n:]:
            print(line)
    else:
        print('[!] app.log chua ton tai')


print('Helpers loaded: pp(), run_query(), parse_step(), show_app_log()')

Helpers loaded: pp(), run_query(), parse_step(), show_app_log()


---
## 3. Test: Single-Intent (Backward Compat)

Kiem tra rang cac query don intent van hoat dong binh thuong sau refactor.

In [4]:
print("\n" + "=" * 70)
print("  TC1: Single Intent — Chat")
print("=" * 70)

result_chat = run_query(orch, "Chao ban, ban la ai?", user_id="e2e_tester")
pp(result_chat, "TC1 RESULT — Chat")

[15:52:02] INFO    | ============================================================
[15:52:02] INFO    | QUERY [49a952f2] (user=e2e_tester): 'Chao ban, ban la ai?'



  TC1: Single Intent — Chat


[15:52:03] INFO    | IntentRouter: 1 intent(s) detected — chat(-)
[15:52:03] INFO    | IntentRouter (1.11s): 1 intent(s) — chat(-)
[15:52:03] INFO    | No current session, creating new
[15:52:03] INFO    | New session created: id=e7bf3c42, topic='', intent=chat, book=None
[15:52:03] INFO    | Session: id=e7bf3c42, user=e2e_tester, topic='', msgs=0
[15:52:03] INFO    | ActionPlanner: 1 plan(s) — chat
[15:52:03] INFO    | Book: ui=KNTT, llm=None, session=KNTT -> effective=KNTT
[15:52:04] INFO    | Total time: 2.13s
[15:52:04] INFO    | ============================================================



────────────────────────────────────────────────────────────
  TC1 RESULT — Chat
────────────────────────────────────────────────────────────
{
  "meta": {
    "request_id": "49a952f2",
    "user_id": "e2e_tester",
    "query": "Chao ban, ban la ai?",
    "ui_book": "KNTT",
    "timestamp": "2026-04-14 15:52:02",
    "total_time_s": 2.13
  },
  "pipeline": {
    "1_context_analyzer": {
      "enriched": false,
      "rewrite": null
    },
    "2_intent_router": {
      "total_intents": 1,
      "intents": [
        {
          "intent": "chat",
          "task_type": null,
          "topic": null,
          "is_new_topic": false,
          "book": null
        }
      ],
      "time_s": 1.11
    },
    "3_session_manager": {
      "session_id": "e7bf3c42",
      "user_id": "e2e_tester",
      "topic": "",
      "intent": "chat",
      "book": null,
      "total_messages": 0,
      "has_quiz_state": false,
      "has_slide_state": false
    },
    "4_action_planner": {
      "total_pla

In [5]:
print("\n" + "=" * 70)
print("  TC2: Single Intent — Generate MCQ")
print("=" * 70)

result_mcq = run_query(orch, "Tao 2 cau trac nghiem ve mang may tinh KNTT lop 11", user_id="e2e_tester")
pp(result_mcq, "TC2 RESULT — Generate MCQ")

[15:52:04] INFO    | ============================================================
[15:52:04] INFO    | QUERY [9822cf35] (user=e2e_tester): 'Tao 2 cau trac nghiem ve mang may tinh KNTT lop 11'



  TC2: Single Intent — Generate MCQ


[15:52:05] INFO    | IntentRouter: 1 intent(s) detected — generate(mcq)
[15:52:05] INFO    | IntentRouter (0.82s): 1 intent(s) — generate(mcq)
[15:52:05] INFO    | Topic changed: '' -> 'mạng máy tính', creating new session
[15:52:05] INFO    | New session created: id=2b2b6138, topic='mạng máy tính', intent=generate, book=KNTT
[15:52:05] INFO    | Session: id=2b2b6138, user=e2e_tester, topic='mạng máy tính', msgs=0
[15:52:05] INFO    | ActionPlanner: 1 plan(s) — generate_quiz
[15:52:05] INFO    | Book: ui=KNTT, llm=KNTT, session=KNTT -> effective=KNTT
[15:52:05] INFO    | RAGAgent: book filter='KNTT' → 1144 chunks in scope
[15:52:05] INFO    | RAGAgent: strategy=hierarchical | grade=None | topic=mạng máy tính | book=KNTT | Query cụ thể + context (grade=None, topic=True) → HRAG


Loading embedding model: dangvantuan/vietnamese-document-embedding...
Model loaded on cuda


[15:52:16] INFO    | HRAG Phase 1: 594 parents searched → 3 selected → 3 unique lessons
[15:52:16] INFO    | HRAG Phase 2: scoped search on 18 child chunks


Loading reranker: AITeamVN/Vietnamese_Reranker...
Reranker loaded on cuda


[15:52:29] INFO    | RAGAgent done: 5 chunks, 23.34s
[15:52:29] INFO    | RAG Search: 5 chunks (23.34s)
[15:52:29] INFO    | Generate: type=mcq, num=3
[15:52:31] INFO    | Handler.handle() -> 2.05s (attempt 1)
[15:52:33] INFO    | Validator: all_valid=True, approved=3 (2.37s)
[15:52:33] INFO    | Saved 3 questions to round 0
[15:52:33] INFO    | Total time: 28.63s
[15:52:33] INFO    | ============================================================



────────────────────────────────────────────────────────────
  TC2 RESULT — Generate MCQ
────────────────────────────────────────────────────────────
{
  "meta": {
    "request_id": "9822cf35",
    "user_id": "e2e_tester",
    "query": "Tao 2 cau trac nghiem ve mang may tinh KNTT lop 11",
    "ui_book": "KNTT",
    "timestamp": "2026-04-14 15:52:04",
    "total_time_s": 28.63
  },
  "pipeline": {
    "1_context_analyzer": {
      "enriched": false,
      "rewrite": null
    },
    "2_intent_router": {
      "total_intents": 1,
      "intents": [
        {
          "intent": "generate",
          "task_type": "mcq",
          "topic": "mạng máy tính",
          "is_new_topic": true,
          "book": "KNTT"
        }
      ],
      "time_s": 0.82
    },
    "3_session_manager": {
      "session_id": "2b2b6138",
      "user_id": "e2e_tester",
      "topic": "mạng máy tính",
      "intent": "generate",
      "book": "KNTT",
      "total_messages": 0,
      "has_quiz_state": false,
     

In [6]:
print("\n" + "=" * 70)
print("  TC3: Single Intent — Explain")
print("=" * 70)

result_explain = run_query(orch, "Giai thich khai niem mang LAN", user_id="e2e_tester")
pp(result_explain, "TC3 RESULT — Explain")

[15:52:33] INFO    | ============================================================
[15:52:33] INFO    | QUERY [e469751b] (user=e2e_tester): 'Giai thich khai niem mang LAN'



  TC3: Single Intent — Explain


[15:52:34] INFO    | IntentRouter: 1 intent(s) detected — explain(-)
[15:52:34] INFO    | IntentRouter (1.08s): 1 intent(s) — explain(-)
[15:52:34] INFO    | Session: id=2b2b6138, user=e2e_tester, topic='mạng máy tính', msgs=2
[15:52:34] INFO    | ActionPlanner: 1 plan(s) — explain_concept
[15:52:34] INFO    | Book: ui=KNTT, llm=None, session=KNTT -> effective=KNTT
[15:52:34] INFO    | RAGAgent: book filter='KNTT' → 1144 chunks in scope
[15:52:34] INFO    | RAGAgent: strategy=broad | grade=None | topic=mạng LAN | book=KNTT | Query tổng quát: broad=False, grade_only=False, topic_broad=True
[15:52:34] INFO    | RAGAgent done: 30 chunks, 0.02s
[15:52:39] INFO    | Total time: 6.2s
[15:52:39] INFO    | ============================================================



────────────────────────────────────────────────────────────
  TC3 RESULT — Explain
────────────────────────────────────────────────────────────
{
  "meta": {
    "request_id": "e469751b",
    "user_id": "e2e_tester",
    "query": "Giai thich khai niem mang LAN",
    "ui_book": "KNTT",
    "timestamp": "2026-04-14 15:52:33",
    "total_time_s": 6.2
  },
  "pipeline": {
    "1_context_analyzer": {
      "enriched": false,
      "rewrite": null
    },
    "2_intent_router": {
      "total_intents": 1,
      "intents": [
        {
          "intent": "explain",
          "task_type": null,
          "topic": "mạng LAN",
          "is_new_topic": false,
          "book": null
        }
      ],
      "time_s": 1.08
    },
    "3_session_manager": {
      "session_id": "2b2b6138",
      "user_id": "e2e_tester",
      "topic": "mạng máy tính",
      "intent": "generate",
      "book": "KNTT",
      "total_messages": 2,
      "has_quiz_state": true,
      "has_slide_state": false
    },
    

---
## 4. Test: Multi-Intent (Core Feature)

Kiem tra kha nang phan tach nhieu y dinh trong 1 cau va thuc thi tuan tu.

In [7]:
# Tao orchestrator moi de tranh state cu
orch2 = Orchestrator(retriever=searcher, reranker=reranker)

print("\n" + "=" * 70)
print("  TC4: Multi-Intent — Explain + Generate MCQ")
print("  Query: 'Giai thich mang LAN la gi roi tao cho toi 3 cau trac nghiem'")
print("  Expected: 2 intents [explain, generate/mcq]")
print("=" * 70)

result_multi_1 = run_query(
    orch2,
    "Giai thich mang LAN la gi roi tao cho toi 3 cau trac nghiem",
    user_id="multi_tester"
)
pp(result_multi_1, "TC4 RESULT — Explain + Generate MCQ")

[15:52:48] INFO    | ============================================================
[15:52:48] INFO    | QUERY [9316143e] (user=multi_tester): 'Giai thich mang LAN la gi roi tao cho toi 3 cau trac nghiem'



  TC4: Multi-Intent — Explain + Generate MCQ
  Query: 'Giai thich mang LAN la gi roi tao cho toi 3 cau trac nghiem'
  Expected: 2 intents [explain, generate/mcq]


[15:52:49] INFO    | IntentRouter: 2 intent(s) detected — explain(-), generate(mcq)
[15:52:49] INFO    | IntentRouter (1.05s): 2 intent(s) — explain(-), generate(mcq)
[15:52:49] INFO    | No current session, creating new
[15:52:49] INFO    | New session created: id=8ef28877, topic='mạng LAN', intent=explain, book=None
[15:52:49] INFO    | Session: id=8ef28877, user=multi_tester, topic='mạng LAN', msgs=0
[15:52:49] INFO    | ActionPlanner: 2 plan(s) — explain_concept, generate_quiz
[15:52:49] INFO    | Book: ui=KNTT, llm=None, session=KNTT -> effective=KNTT
[15:52:49] INFO    | RAGAgent: book filter='KNTT' → 1144 chunks in scope
[15:52:49] INFO    | RAGAgent: strategy=broad | grade=None | topic=mạng LAN | book=KNTT | Query tổng quát: broad=False, grade_only=False, topic_broad=True
[15:52:49] INFO    | RAGAgent done: 30 chunks, 0.00s
[15:52:54] INFO    | RAGAgent: book filter='KNTT' → 1144 chunks in scope
[15:52:54] INFO    | RAGAgent: strategy=hierarchical | grade=None | topic=mạng LAN 


────────────────────────────────────────────────────────────
  TC4 RESULT — Explain + Generate MCQ
────────────────────────────────────────────────────────────
{
  "meta": {
    "request_id": "9316143e",
    "user_id": "multi_tester",
    "query": "Giai thich mang LAN la gi roi tao cho toi 3 cau trac nghiem",
    "ui_book": "KNTT",
    "timestamp": "2026-04-14 15:52:48",
    "total_time_s": 12.74
  },
  "pipeline": {
    "1_context_analyzer": {
      "enriched": false,
      "rewrite": null
    },
    "2_intent_router": {
      "total_intents": 2,
      "intents": [
        {
          "intent": "explain",
          "task_type": null,
          "topic": "mạng LAN",
          "is_new_topic": true,
          "book": null
        },
        {
          "intent": "generate",
          "task_type": "mcq",
          "topic": "mạng LAN",
          "is_new_topic": false,
          "book": null
        }
      ],
      "time_s": 1.05
    },
    "3_session_manager": {
      "session_id": "8ef28

In [8]:
orch3 = Orchestrator(retriever=searcher, reranker=reranker)

print("\n" + "=" * 70)
print("  TC5: Multi-Intent — Generate MCQ + Generate Essay")
print("  Query: 'Tao 3 cau trac nghiem va 2 cau tu luan ve he dieu hanh'")
print("  Expected: 2 intents [generate/mcq, generate/essay]")
print("=" * 70)

result_multi_2 = run_query(
    orch3,
    "Tao 3 cau trac nghiem va 2 cau tu luan ve he dieu hanh",
    user_id="multi_tester_2"
)
pp(result_multi_2, "TC5 RESULT — Generate MCQ + Essay")

[15:53:10] INFO    | ============================================================
[15:53:10] INFO    | QUERY [6bd095f4] (user=multi_tester_2): 'Tao 3 cau trac nghiem va 2 cau tu luan ve he dieu hanh'



  TC5: Multi-Intent — Generate MCQ + Generate Essay
  Query: 'Tao 3 cau trac nghiem va 2 cau tu luan ve he dieu hanh'
  Expected: 2 intents [generate/mcq, generate/essay]


[15:53:12] INFO    | IntentRouter: 2 intent(s) detected — generate(mcq), generate(essay)
[15:53:12] INFO    | IntentRouter (1.34s): 2 intent(s) — generate(mcq), generate(essay)
[15:53:12] INFO    | No current session, creating new
[15:53:12] INFO    | New session created: id=390dfee9, topic='hệ điều hành', intent=generate, book=None
[15:53:12] INFO    | Session: id=390dfee9, user=multi_tester_2, topic='hệ điều hành', msgs=0
[15:53:12] INFO    | ActionPlanner: 1 plan(s) — generate_quiz
[15:53:12] INFO    | Book: ui=KNTT, llm=None, session=KNTT -> effective=KNTT
[15:53:12] INFO    | RAGAgent: book filter='KNTT' → 1144 chunks in scope
[15:53:12] INFO    | RAGAgent: strategy=hierarchical | grade=None | topic=hệ điều hành | book=KNTT | Query cụ thể + context (grade=None, topic=True) → HRAG
[15:53:12] INFO    | HRAG Phase 1: 594 parents searched → 3 selected → 3 unique lessons
[15:53:12] INFO    | HRAG Phase 2: scoped search on 13 child chunks
[15:53:17] INFO    | RAGAgent done: 5 chunks, 5.


────────────────────────────────────────────────────────────
  TC5 RESULT — Generate MCQ + Essay
────────────────────────────────────────────────────────────
{
  "meta": {
    "request_id": "6bd095f4",
    "user_id": "multi_tester_2",
    "query": "Tao 3 cau trac nghiem va 2 cau tu luan ve he dieu hanh",
    "ui_book": "KNTT",
    "timestamp": "2026-04-14 15:53:10",
    "total_time_s": 13.42
  },
  "pipeline": {
    "1_context_analyzer": {
      "enriched": false,
      "rewrite": null
    },
    "2_intent_router": {
      "total_intents": 2,
      "intents": [
        {
          "intent": "generate",
          "task_type": "mcq",
          "topic": "hệ điều hành",
          "is_new_topic": true,
          "book": null
        },
        {
          "intent": "generate",
          "task_type": "essay",
          "topic": "hệ điều hành",
          "is_new_topic": false,
          "book": null
        }
      ],
      "time_s": 1.34
    },
    "3_session_manager": {
      "session_id":

---
## 5. Summary Report

Gom tat ca ket qua, validate va luu JSON report.

In [9]:
all_results = [
    ("TC1: Chat", 1, result_chat),
    ("TC2: Generate MCQ", 1, result_mcq),
    ("TC3: Explain", 1, result_explain),
    ("TC4: Explain+MCQ", 2, result_multi_1),
    ("TC5: MCQ+Essay", 2, result_multi_2),
]

summary_rows = []
for name, expected, result in all_results:
    detected = result["pipeline"]["2_intent_router"]["total_intents"]
    planned = result["pipeline"]["4_action_planner"]["total_plans"]
    match = detected >= expected  # >= vi LLM co the phat hien dung hoac nhieu hon
    summary_rows.append({
        "test": name,
        "expected_intents": expected,
        "detected_intents": detected,
        "planned_actions": planned,
        "match": "PASS" if match else "FAIL",
        "time_s": result["meta"]["total_time_s"],
        "response_sections": result["response"]["num_sections"],
        "response_length": result["response"]["length"],
    })

summary = {
    "total_tests": len(all_results),
    "passed": sum(1 for r in summary_rows if r["match"] == "PASS"),
    "failed": sum(1 for r in summary_rows if r["match"] == "FAIL"),
    "results": summary_rows,
}

pp(summary, "FINAL SUMMARY")

# Save full report
full_report = {
    "summary": summary,
    "details": {
        name: result for name, _, result in all_results
    },
}
report_path = PROJECT_ROOT / "logs" / "multi_intent_e2e_report.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(
    json.dumps(full_report, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
print(f"\nFull report saved: {report_path}")


────────────────────────────────────────────────────────────
  FINAL SUMMARY
────────────────────────────────────────────────────────────
{
  "total_tests": 5,
  "passed": 5,
  "failed": 0,
  "results": [
    {
      "test": "TC1: Chat",
      "expected_intents": 1,
      "detected_intents": 1,
      "planned_actions": 1,
      "match": "PASS",
      "time_s": 2.13,
      "response_sections": 1,
      "response_length": 247
    },
    {
      "test": "TC2: Generate MCQ",
      "expected_intents": 1,
      "detected_intents": 1,
      "planned_actions": 1,
      "match": "PASS",
      "time_s": 28.63,
      "response_sections": 1,
      "response_length": 789
    },
    {
      "test": "TC3: Explain",
      "expected_intents": 1,
      "detected_intents": 1,
      "planned_actions": 1,
      "match": "PASS",
      "time_s": 6.2,
      "response_sections": 47,
      "response_length": 4444
    },
    {
      "test": "TC4: Explain+MCQ",
      "expected_intents": 2,
      "detected_intent

---
## 6. Debug — App Log (last 30 lines)

In [10]:
show_app_log(40)

[2026-04-14 15:52:49] INFO    | chatbot | Book: ui=KNTT, llm=None, session=KNTT -> effective=KNTT
[2026-04-14 15:52:49] INFO    | chatbot | RAGAgent: book filter='KNTT' → 1144 chunks in scope
[2026-04-14 15:52:49] INFO    | chatbot | RAGAgent: strategy=broad | grade=None | topic=mạng LAN | book=KNTT | Query tổng quát: broad=False, grade_only=False, topic_broad=True
[2026-04-14 15:52:49] INFO    | chatbot | RAGAgent done: 30 chunks, 0.00s
[2026-04-14 15:52:54] INFO    | chatbot | RAGAgent: book filter='KNTT' → 1144 chunks in scope
[2026-04-14 15:52:54] INFO    | chatbot | RAGAgent: strategy=hierarchical | grade=None | topic=mạng LAN | book=KNTT | Query cụ thể + context (grade=None, topic=True) → HRAG
[2026-04-14 15:52:54] INFO    | chatbot | HRAG Phase 1: 594 parents searched → 3 selected → 3 unique lessons
[2026-04-14 15:52:54] INFO    | chatbot | HRAG Phase 2: scoped search on 10 child chunks
[2026-04-14 15:52:57] INFO    | chatbot | RAGAgent done: 5 chunks, 3.05s
[2026-04-14 15:52:57